# Live Persian streaming on a Colab T4

Run this repo's WhisperLiveKit streaming stack against the private
[`ameerhossein/liquidwhisper-neyshekar-ct2-gpu`](https://huggingface.co/ameerhossein/liquidwhisper-neyshekar-ct2-gpu)
CTranslate2 model and use the microphone page through a temporary Cloudflare tunnel.

## Before you run

1. **Runtime -> Change runtime type -> T4 GPU.**
2. Add a Colab Secret named **`HF_TOKEN`** with read access to the private model repo
   (Tools -> Secrets). Do not paste the token into a cell.
3. **Runtime -> Run all.**
4. Open the `https://....trycloudflare.com` URL printed by the tunnel cell and allow
   microphone access.
5. When finished, tick `STOP_SERVER` in the final cell and run it.

> The tunnel URL is public while the notebook runs. Anyone with the link can use the
> page. The final stop cell is disabled by default so Run all leaves the demo running.

The first run installs the pinned runtime and downloads the model inside Colab. That can
take several minutes; later cells reuse the files for the life of the session.

To compare decoding settings, change `WLK_BEAM`, `WLK_TEMPERATURE`, or `ASR_COALESCE`
in the Configuration cell, **re-run Configuration**, then re-run the **Start server** and
**Open tunnel** cells.


In [ ]:
#@title Configuration { display-mode: "form" }
REPO_URL = "https://github.com/ameerhossein/LiquidWhisperLiveKit.git" #@param {type:"string"}
REPO_REF = "main" #@param {type:"string"}
HF_REPO_ID = "ameerhossein/liquidwhisper-neyshekar-ct2-gpu" #@param {type:"string"}
HF_REVISION = "" #@param {type:"string"}
HF_SUBFOLDER = "" #@param {type:"string"}
WLK_BEAM = 1 #@param {type:"integer"}
WLK_TEMPERATURE = "0.0" #@param ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0", ""] {type:"string"}
ASR_COALESCE = "0" #@param {type:"string"}
PORT = 8000 #@param {type:"integer"}

from pathlib import Path

PROJECT_DIR = Path("/content/LiquidWhisperLiveKit")
STATE_DIR = Path("/content/livebuzzasr-colab")
STATE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR_FILE = STATE_DIR / "model-dir.txt"
SERVER_LOG = STATE_DIR / "server.log"
TUNNEL_LOG = STATE_DIR / "cloudflared.log"
SERVER_PID = STATE_DIR / "server.pid"
TUNNEL_PID = STATE_DIR / "tunnel.pid"
CLOUDFLARED = Path("/content/bin/cloudflared")

print("Configuration loaded")
print("  project:", PROJECT_DIR)
print("  model:  ", HF_REPO_ID, HF_SUBFOLDER or "<auto-detect>")
print("  decode: ", {"beam": WLK_BEAM, "temperature": WLK_TEMPERATURE, "coalesce": ASR_COALESCE})


In [ ]:
import shutil
import subprocess

import torch

smi = shutil.which("nvidia-smi")
if smi:
    print(subprocess.run(
        [smi, "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip() or "nvidia-smi produced no output")
else:
    print("nvidia-smi not found")
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda,
      "| available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU detected. Select Runtime > Change runtime type > T4 GPU, "
        "then Runtime > Restart session and run again."
    )

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
free, total = torch.cuda.mem_get_info()
print(f"GPU: {name} (compute {major}.{minor}) | "
      f"VRAM free/total: {free / 2**30:.1f}/{total / 2**30:.1f} GiB")

if major < 7:
    raise SystemExit(
        "This GPU is too old for float16 inference (compute capability < 7.0). "
        "Use a T4 runtime."
    )
if "T4" not in name:
    print("Warning: this is not a T4. Continuing on any compute capability >= 7.0 GPU.")

disk = shutil.disk_usage("/content")
print(f"Disk free: {disk.free / 2**30:.1f} GiB")
if disk.free < 12 * 2**30:
    raise SystemExit(
        "At least 12 GiB free disk is required for the venv, CUDA wheels, and model. "
        "Free space and re-run."
    )


In [ ]:
import subprocess

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "fetch", "--depth", "1", "origin", REPO_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "--force", "FETCH_HEAD"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "clean", "-ffdx",
         "-e", ".venv", "-e", ".cache", "-e", "models"],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise SystemExit(
        f"{PROJECT_DIR} exists but is not a git checkout. Delete it and re-run this cell."
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "log", "-1", "--oneline"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("HEAD:", head)


In [ ]:
import os
import sys
from getpass import getpass
from urllib.parse import quote

import requests

token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

token = token or os.environ.get("HF_TOKEN")
if not token:
    token = getpass("HF token with read access to the private model: ")
if not token:
    raise SystemExit(
        "HF_TOKEN is required. Add a Colab Secret named HF_TOKEN "
        "(Tools > Secrets), then re-run this cell."
    )
os.environ["HF_TOKEN"] = token

sys.path.insert(0, str(PROJECT_DIR / "scripts"))
from colab_support import find_ct2_subfolder

api = f"https://huggingface.co/api/models/{HF_REPO_ID}"
headers = {"Authorization": f"Bearer {token}"}
meta = requests.get(api, headers=headers, timeout=30)
if meta.status_code in (401, 403):
    raise SystemExit(
        "HF_TOKEN was rejected (401/403). Check that it is valid and has read "
        "access to the private model repo."
    )
if meta.status_code == 404:
    raise SystemExit(
        f"Model repo not found or not visible to this token (404): {HF_REPO_ID}. "
        "Check HF_TOKEN access and the repo id."
    )
meta.raise_for_status()
print("Repo:", meta.json().get("id"), "| private:", meta.json().get("private"))

revision = HF_REVISION or "main"
tree_url = f"{api}/tree/{quote(revision, safe='')}?recursive=true"
tree = requests.get(tree_url, headers=headers, timeout=60)
if tree.status_code in (401, 403, 404):
    raise SystemExit(
        f"Cannot read revision {revision!r} of {HF_REPO_ID} "
        f"({tree.status_code}). Check HF_REVISION and HF_TOKEN access."
    )
tree.raise_for_status()
files = [item["path"] for item in tree.json() if item.get("type") == "file"]
if not files:
    raise SystemExit("The model repo/revision returned no files.")

subfolder = find_ct2_subfolder(files, HF_SUBFOLDER)
os.environ["HF_SUBFOLDER"] = subfolder
print("CT2 subfolder:", subfolder or "<repo root>")
print("Model files found:", len(files))


In [ ]:
import os
import shutil
import subprocess
import sys

print("Installing uv into the Colab system Python...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
uv = shutil.which("uv")
if not uv:
    raise SystemExit("uv was installed but is not on PATH. Restart the runtime and run again.")
print(subprocess.run([uv, "--version"], capture_output=True, text=True, check=True).stdout.strip())

env = os.environ.copy()
env.update({
    "DEVICE": "gpu",
    "HF_REPO_ID": HF_REPO_ID,
    "HF_REVISION": HF_REVISION,
    "HF_SUBFOLDER": subfolder,
    "HF_TOKEN": token,
    "HF_HOME": str(PROJECT_DIR / ".cache" / "hf"),
    "HF_HUB_CACHE": str(PROJECT_DIR / ".cache" / "hf" / "hub"),
    "UV_CACHE_DIR": str(PROJECT_DIR / ".cache" / "uv"),
    "UV_PYTHON_INSTALL_DIR": str(PROJECT_DIR / ".cache" / "uv-python"),
    "XDG_CACHE_HOME": str(PROJECT_DIR / ".cache" / "xdg"),
    "TORCH_HOME": str(PROJECT_DIR / ".cache" / "torch"),
})

print("Building the pinned runtime and downloading the model (several minutes)...")
subprocess.run(["bash", "setup.sh"], cwd=PROJECT_DIR, env=env, check=True)

venv_python = PROJECT_DIR / ".venv" / "bin" / "python"
if not venv_python.is_file():
    raise SystemExit(f"setup.sh finished without creating {venv_python}")

resolved = subprocess.run(
    [str(venv_python), "scripts/settings.py", "--resolve-model"],
    cwd=PROJECT_DIR, env=env, capture_output=True, text=True, check=True,
).stdout.strip().splitlines()
if not resolved:
    raise SystemExit("Could not resolve the downloaded model directory.")
model_dir = Path(resolved[-1])
MODEL_DIR_FILE.write_text(str(model_dir), encoding="utf-8")
for key in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
    os.environ.pop(key, None)
print("Model directory:", model_dir)
for name in ("model.bin", "config.json", "vocabulary.json", "tokenizer.json", "preprocessor_config.json"):
    file = model_dir / name
    size = file.stat().st_size / 2**20 if file.is_file() else 0
    print(f"  {name}: {size:.1f} MiB")


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

sys.path.insert(0, str(PROJECT_DIR / "scripts"))
from colab_support import find_cudnn_library, nvidia_library_dirs

venv_python = PROJECT_DIR / ".venv" / "bin" / "python"
model_dir = Path(MODEL_DIR_FILE.read_text(encoding="utf-8").strip())
if not model_dir.is_dir():
    raise SystemExit(f"Persisted model directory does not exist: {model_dir}. Re-run the setup cell.")

uv = shutil.which("uv") or "uv"

def cuda_env():
    site = subprocess.run(
        [str(venv_python), "-c", "import site; print(site.getsitepackages()[0])"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    libs = nvidia_library_dirs(site)
    env = os.environ.copy()
    if libs:
        env["LD_LIBRARY_PATH"] = os.pathsep.join(
            libs + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])
        )
    return env, libs, site

env, libs, site = cuda_env()
print("NVIDIA libraries:", libs or "<none>")
cudnn = find_cudnn_library(site)
if not cudnn:
    print("cuDNN 9 was not found; installing it into the venv...")
    subprocess.run(
        [uv, "pip", "install", "--python", str(venv_python),
         "nvidia-cudnn-cu12==9.*"],
        check=True,
    )
    env, libs, site = cuda_env()
    cudnn = find_cudnn_library(site)
    print("NVIDIA libraries:", libs or "<none>")
if not cudnn:
    raise SystemExit(
        "cuDNN 9 (libcudnn_ops.so.9) is still missing after installation. "
        "Restart the runtime and run from the top."
    )
print("cuDNN 9:", cudnn)

preflight = '''
import ctranslate2
import torch
print("torch", torch.__version__, "cuda", torch.version.cuda,
      "available", torch.cuda.is_available())
types = ctranslate2.get_supported_compute_types("cuda")
print("CT2 CUDA compute types:", types)
assert torch.cuda.is_available(), "CUDA is not available to torch"
assert "float16" in types, f"float16 is not supported: {types}"
'''

for attempt in (1, 2):
    probe = subprocess.run(
        [str(venv_python), "-c", preflight],
        env=env, capture_output=True, text=True,
    )
    print(probe.stdout.strip())
    if probe.returncode == 0:
        break
    error = probe.stderr.strip()
    print(error[-4000:])
    if attempt == 1 and any(
        word in error.lower() for word in ("cudnn", "float16", "compute type")
    ):
        print("cuDNN 9 is missing or float16 is unavailable; installing and retrying...")
        subprocess.run(
            [uv, "pip", "install", "--python", str(venv_python),
             "nvidia-cudnn-cu12==9.*"],
            check=True,
        )
        env, libs, site = cuda_env()
        cudnn = find_cudnn_library(site)
        print("NVIDIA libraries:", libs or "<none>")
        print("cuDNN 9:", cudnn or "<missing>")
        continue
    raise SystemExit("CTranslate2 CUDA preflight failed. See the error above.")

validation = subprocess.run(
    [str(venv_python), "scripts/settings.py", "--check-model", str(model_dir)],
    cwd=PROJECT_DIR, env=env, capture_output=True, text=True,
)
print(validation.stdout.strip())
if validation.returncode != 0:
    raise SystemExit(validation.stderr.strip() or "Model validation failed.")
print("Preflight OK:", model_dir)


In [ ]:
import os
import signal
import subprocess
import time
from pathlib import Path

import requests

from colab_support import nvidia_library_dirs, stop_process

stop_process(SERVER_PID, ("run.sh", "scripts/serve.py"))
SERVER_LOG.unlink(missing_ok=True)

venv_python = PROJECT_DIR / ".venv" / "bin" / "python"
model_dir = Path(MODEL_DIR_FILE.read_text(encoding="utf-8").strip())
site = subprocess.run(
    [str(venv_python), "-c", "import site; print(site.getsitepackages()[0])"],
    capture_output=True, text=True, check=True,
).stdout.strip()
libs = nvidia_library_dirs(site)

env = os.environ.copy()
for key in ("HF_REPO_ID", "HF_REVISION", "HF_SUBFOLDER"):
    env.pop(key, None)
env.pop("HF_TOKEN", None)
env.pop("HUGGING_FACE_HUB_TOKEN", None)
env.update({
    "DEVICE": "gpu",
    "MODEL_DIR": str(model_dir),
    "HOST": "0.0.0.0",
    "PORT": str(PORT),
    "WLK_BEAM": str(WLK_BEAM),
    "WLK_TEMPERATURE": str(WLK_TEMPERATURE),
    "ASR_COALESCE": str(ASR_COALESCE),
    "WLK_PROFILE": "tuned",
})
print("Applied decoding:", {
    "beam": WLK_BEAM,
    "temperature": WLK_TEMPERATURE,
    "coalesce": ASR_COALESCE,
})
if libs:
    env["LD_LIBRARY_PATH"] = os.pathsep.join(
        libs + ([env["LD_LIBRARY_PATH"]] if env.get("LD_LIBRARY_PATH") else [])
    )

log = SERVER_LOG.open("w", encoding="utf-8")
process = subprocess.Popen(
    ["bash", "run.sh", "--pcm-input"],
    cwd=PROJECT_DIR,
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
log.close()
SERVER_PID.write_text(str(process.pid), encoding="utf-8")
print("Server PID:", process.pid)
print("Waiting for the model and HTTP server...")

health = None
deadline = time.time() + 240
while time.time() < deadline:
    if process.poll() is not None:
        print(SERVER_LOG.read_text(encoding="utf-8", errors="ignore")[-8000:])
        raise SystemExit(f"Server exited with code {process.returncode}. See the log above.")
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2)
        if response.ok:
            health = response.json()
            if health.get("ready"):
                break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    print(SERVER_LOG.read_text(encoding="utf-8", errors="ignore")[-8000:])
    raise SystemExit("Server did not become ready within 240 seconds.")

print("Health:", health)
print(f"Local UI: http://127.0.0.1:{PORT}")


In [ ]:
import os
import subprocess

venv_python = PROJECT_DIR / ".venv" / "bin" / "python"
test_env = os.environ.copy()
test_env.pop("HF_TOKEN", None)
test_env.pop("HUGGING_FACE_HUB_TOKEN", None)
print("Streaming testdata/verify-fa.wav through the local WebSocket...")
subprocess.run(
    [
        str(venv_python),
        "scripts/stream_test.py",
        "testdata/verify-fa.wav",
        "--url", f"ws://127.0.0.1:{PORT}/asr",
        "--realtime", "1.0",
        "--timeout", "300",
        "--require-text",
    ],
    cwd=PROJECT_DIR,
    env=test_env,
    check=True,
)


In [ ]:
import os
import signal
import subprocess
import time

import requests
from IPython.display import HTML, display

from colab_support import health_payload_is_ready, parse_tunnel_url, stop_process

stop_process(TUNNEL_PID, ("cloudflared",))
TUNNEL_LOG.unlink(missing_ok=True)
CLOUDFLARED.parent.mkdir(parents=True, exist_ok=True)
if not CLOUDFLARED.is_file():
    print("Downloading cloudflared...")
    subprocess.run(
        ["curl", "-L", "--fail", "--retry", "3",
         "-o", str(CLOUDFLARED),
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True,
    )
    CLOUDFLARED.chmod(0o755)

tunnel_env = os.environ.copy()
tunnel_env.pop("HF_TOKEN", None)
tunnel_env.pop("HUGGING_FACE_HUB_TOKEN", None)

log = TUNNEL_LOG.open("w", encoding="utf-8")
process = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    env=tunnel_env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
log.close()
TUNNEL_PID.write_text(str(process.pid), encoding="utf-8")

url = None
deadline = time.time() + 90
while time.time() < deadline:
    if process.poll() is not None:
        print(TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")[-8000:])
        raise SystemExit(f"cloudflared exited with code {process.returncode}. See the log above.")
    url = parse_tunnel_url(TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore"))
    if url:
        break
    time.sleep(1)

if not url:
    print(TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")[-8000:])
    raise SystemExit("cloudflared did not print a trycloudflare.com URL within 90 seconds.")

print("Tunnel URL:", url)

tunnel_health = None
for attempt in range(30):
    try:
        response = requests.get(f"{url}/health", timeout=10)
        if response.ok:
            try:
                payload = response.json()
            except ValueError:
                payload = None
            if health_payload_is_ready(payload):
                tunnel_health = payload
                print("Tunnel health:", payload)
                break
    except requests.RequestException:
        pass
    time.sleep(2)

if tunnel_health:
    display(HTML(f'<a href="{url}" target="_blank">{url}</a>'))
else:
    print(TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")[-8000:])
    print(f"Warning: observed {url} but /health did not report ready: true.")
    print("Open the URL manually or re-run this cell. The link is unverified.")


In [ ]:
#@title Stop the demo { display-mode: "form" }
STOP_SERVER = False #@param {type:"boolean"}
PRINT_LOGS = False #@param {type:"boolean"}

from colab_support import stop_process

if not STOP_SERVER:
    print("Server and tunnel left running.")
    print("Tick STOP_SERVER above and run this cell again to stop them.")
else:
    for pid_file, expected in (
        (TUNNEL_PID, ("cloudflared",)),
        (SERVER_PID, ("run.sh", "scripts/serve.py")),
    ):
        stop_process(pid_file, expected)
    print("Stopped server and tunnel.")

    if PRINT_LOGS:
        for path in (SERVER_LOG, TUNNEL_LOG):
            if path.is_file():
                print(f"\n--- {path.name} (last 20 lines) ---")
                print("\n".join(path.read_text(encoding="utf-8", errors="ignore").splitlines()[-20:]))


## Troubleshooting

- **`No CUDA GPU detected`** - choose Runtime -> Change runtime type -> T4 GPU, then restart the session.
- **HF 401/403/404** - the `HF_TOKEN` Secret is missing, invalid, or lacks access to the private repo.
- **`No complete CTranslate2 directory` / `Multiple complete ...`** - set `HF_SUBFOLDER` to one of the directories named in the error.
- **cuDNN errors** - the preflight installs `nvidia-cudnn-cu12==9.*` and retries once. If it still fails, restart the runtime and run from the top.
- **Tunnel URL does not open** - wait 10-20 seconds and re-run the tunnel cell. If cloudflared exits, read the log it prints.
- **`wlk` prints `Model: base`** - that banner is upstream and misleading; trust `/health` and the self-test.
- **Public URL** - the quick tunnel is reachable by anyone who has the link; run the stop cell when finished.
